In [ ]:
from pathlib import Path
import json

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
CHAPTER = _cwd.parent if (_cwd.parent / "data" / "wildfire").exists() else _cwd
DATA = CHAPTER / "data"
WILDFIRE_DIR = DATA / "wildfire"
PROCESSED = DATA / "processed"
INPUT = DATA / "input"
RESULTS_FIG = CHAPTER / "results" / "figures"
RESULTS_FIG.mkdir(parents=True, exist_ok=True)

# Okabe–Ito (colorblind-safe)
COLOR_ARTICLES = "#0072B2"   # blue
COLOR_LLM = "#0072B2"        # blue
COLOR_WILDFIRE = "#D55E00"   # vermillion (not orange)
TITLE_FS = 14                # default matplotlib title ~12


def save_fig(name: str) -> None:
    path = RESULTS_FIG / name
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved {path}")


print(f"CHAPTER = {CHAPTER}")
print(f"WILDFIRE_DIR exists: {WILDFIRE_DIR.exists()}")
print(f"PROCESSED exists: {PROCESSED.exists()}")


## Wildfire dataset EDA

In [ ]:
# Scan the dataset lazily (takes 0 seconds)
df_lazy = pl.scan_csv(WILDFIRE_DIR / "wildfires_2km_decade_2017-2022_no-military-bases.csv")

print(df_lazy.columns)

# Get a quick glimpse of the schema and first few rows
print(df_lazy.head(5).collect())

missing_data = df_lazy.null_count().collect()
print(missing_data)

# Get basic summary statistics for numeric columns
stats = df_lazy.describe()
print(stats)


In [ ]:
#add all feature

In [ ]:
# Clean and cast string columns to Float64, then collect into memory
df_collected = df_lazy.with_columns([
    pl.col("spi01_mean_mean").cast(pl.Float64, strict=False),
    pl.col("spi03_mean_mean").cast(pl.Float64, strict=False),
    pl.col("spi06_mean_mean").cast(pl.Float64, strict=False),
]).collect()

# 10% random sample for quick plotting
df_sample = df_collected.sample(fraction=0.1, seed=42)

plt.figure(figsize=(10, 8))
plt.scatter(df_sample['x'], df_sample['y'], c=df_sample['vpd_mean_mean'],
            cmap='YlOrRd', s=1, alpha=0.5)
plt.colorbar(label='Vapor Pressure Deficit (VPD)')
plt.title('Spatial Distribution of Atmospheric Aridity (VPD)')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.show()


In [ ]:
plt.figure(figsize=(12, 10))

# Plot the 'No Fire' background points first in a faint gray
df_no_fire = df_sample.filter(pl.col('wildfires_sum') == 0)
plt.scatter(df_no_fire['x'], df_no_fire['y'], color='#E0E0E0', s=2, alpha=0.3, label='No Fire')

# Plot the actual wildfire points with larger sizes and crisp edges
df_fires = df_sample.filter(pl.col('wildfires_sum') > 0)
scatter = plt.scatter(df_fires['x'], df_fires['y'],
                      c=df_fires['wildfires_sum'],
                      cmap='YlOrRd',
                      s=25,
                      alpha=1.0,
                      edgecolors='black',
                      linewidths=0.4)

plt.colorbar(scatter, label='Wildfires (Sum)')
plt.title('Spatial Distribution of Wildfires in the NL', fontsize=14, fontweight='bold')
plt.xlabel('X Coordinate (Rijksdriehoek / Dutch Grid)')
plt.ylabel('Y Coordinate')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()


In [ ]:
# Aggregate wildfire counts by month.
# Prefers the existing `time` column and falls back to `year`/`month` if needed.
if "time" in df_collected.columns:
    monthly_fires = (
        df_collected
        .with_columns(
            pl.col("time")
            .cast(pl.Utf8, strict=False)
            .str.strptime(pl.Datetime, strict=False)
            .dt.truncate("1mo")
            .alias("month")
        )
        .group_by("month")
        .agg(
            pl.col("wildfires_sum").sum().alias("wildfires_total")
        )
        .sort("month")
    )
elif {"year", "month"}.issubset(df_collected.columns):
    monthly_fires = (
        df_collected
        .with_columns(
            pl.date(
                pl.col("year").cast(pl.Int32, strict=False),
                pl.col("month").cast(pl.Int32, strict=False),
                pl.lit(1),
            )
            .cast(pl.Datetime, strict=False)
            .alias("month")
        )
        .group_by("month")
        .agg(
            pl.col("wildfires_sum").sum().alias("wildfires_total")
        )
        .sort("month")
    )
else:
    raise ValueError("Could not find a usable time column. Expected 'time' or 'year'/'month'.")

print(monthly_fires.head(12))

monthly_fires_pd = monthly_fires.to_pandas()

plt.figure(figsize=(14, 6))
plt.plot(
    monthly_fires_pd["month"],
    monthly_fires_pd["wildfires_total"],
    marker="o",
    linewidth=2,
    color=COLOR_WILDFIRE,
)
plt.fill_between(
    monthly_fires_pd["month"],
    monthly_fires_pd["wildfires_total"],
    color=COLOR_WILDFIRE,
    alpha=0.15,
)
plt.title("Monthly Wildfire Count", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Wildfires (sum)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("monthly_wildfire_count.png")
plt.show()


## Article counts

In [ ]:
# Load the deduplicated article dataset and aggregate article counts by month.
print(
    "Note: this section is not important for the thesis figures; "
    "it is only included to keep the full code view."
)
articles_path = Path(
    r"C:\Users\hoeven\Downloads\Modulaire drought impact location classifier"
    r"\Preproccesing\data\output\all_articles_deduplicated.json"
)
articles = json.loads(articles_path.read_text(encoding="utf-8"))

articles_df = pl.from_dicts([
    {
        "publication_date": article.get("features", {}).get("publication_date") or article.get("meta", {}).get("date")
    }
    for article in articles
])

monthly_articles = (
    articles_df
    .with_columns(
        pl.col("publication_date")
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Datetime, strict=False)
        .dt.truncate("1mo")
        .alias("month")
    )
    .drop_nulls("month")
    .group_by("month")
    .agg(pl.len().alias("article_count"))
    .sort("month")
)

print(monthly_articles.head(12))

monthly_articles_pd_full = monthly_articles.to_pandas()
full_months = pd.date_range(
    start=pd.Timestamp('1990-08-01 00:00:00'),
    end=monthly_articles_pd_full["month"].max(),
    freq="MS",
)
monthly_articles_pd_full = (
    monthly_articles_pd_full.set_index("month")
    .reindex(full_months, fill_value=0)
    .rename_axis("month")
    .reset_index()
)

plt.figure(figsize=(14, 6))
plt.plot(
    monthly_articles_pd_full["month"],
    monthly_articles_pd_full["article_count"],
    color=COLOR_ARTICLES,
    linewidth=2,
)
plt.fill_between(
    monthly_articles_pd_full["month"],
    monthly_articles_pd_full["article_count"],
    color=COLOR_ARTICLES,
    alpha=0.15,
)
plt.title("Monthly Article Count (Full History)", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Articles")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("monthly_article_count_full.png")
plt.show()


In [ ]:
# Reindex the already-loaded monthly_articles to the 2017-2022 comparison window
comparison_months_2017 = pd.date_range(
    start=pd.Timestamp('2017-01-01 00:00:00'),
    end=pd.Timestamp('2022-01-01 00:00:00'),
    freq="MS",
)
monthly_articles_pd_2017 = (
    monthly_articles.to_pandas()
    .set_index("month")
    .reindex(comparison_months_2017, fill_value=0)
    .rename_axis("month")
    .reset_index()
)

plt.figure(figsize=(14, 6))
plt.plot(
    monthly_articles_pd_2017["month"],
    monthly_articles_pd_2017["article_count"],
    color=COLOR_ARTICLES,
    linewidth=2,
)
plt.fill_between(
    monthly_articles_pd_2017["month"],
    monthly_articles_pd_2017["article_count"],
    color=COLOR_ARTICLES,
    alpha=0.15,
)
plt.title("Monthly Article Count (2017-2022)", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Articles")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("monthly_article_count_2017_2022.png")
plt.show()


## Articles vs wildfires comparison

In [ ]:
# Compare article counts and wildfire counts on the same monthly axis.
# Both series are normalized to 0-1 so the trend shapes can be compared directly.
start_date = pd.Timestamp("2017-01-01")
end_date = pd.Timestamp("2022-08-01")
comparison_months = pd.date_range(start=start_date, end=end_date, freq="MS")

articles_series = (
    monthly_articles_pd_2017.set_index("month")["article_count"]
    .reindex(comparison_months, fill_value=0)
)
fires_series = (
    monthly_fires_pd.set_index("month")["wildfires_total"]
    .reindex(comparison_months, fill_value=0)
)

def normalize(series):
    series_min = series.min()
    series_max = series.max()
    if series_max == series_min:
        return series * 0
    return (series - series_min) / (series_max - series_min)

comparison_df = pd.DataFrame({
    "month": comparison_months,
    "articles_scaled": normalize(articles_series).to_numpy(),
    "fires_scaled": normalize(fires_series).to_numpy(),
    "article_count": articles_series.to_numpy(),
    "fire_count": fires_series.to_numpy(),
})

plt.figure(figsize=(14, 6))
plt.plot(
    comparison_df["month"],
    comparison_df["articles_scaled"],
    label="Articles (scaled)",
    color=COLOR_ARTICLES,
    linewidth=2.2,
    linestyle="-",
)
plt.plot(
    comparison_df["month"],
    comparison_df["fires_scaled"],
    label="Wildfires (scaled)",
    color=COLOR_WILDFIRE,
    linewidth=2.2,
    linestyle="--",
)
plt.title("Monthly Article and Wildfire Trends", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Scaled value (0-1)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
save_fig("monthly_article_vs_wildfire_trends.png")
plt.show()


## LLM wildfire occurrences vs wildfire dataset


In [ ]:
# Load LLM-enriched articles and select Wildfire Occurrence impacts.
llm_articles_path = INPUT / "impacts_for_geocoding.json"
llm_articles = json.loads(llm_articles_path.read_text(encoding="utf-8"))

WILDFIRE_CLASS = "Wildfire Occurrence"
wildfire_rows = []
for article in llm_articles:
    publication_date = (
        article.get("features", {}).get("publication_date")
        or article.get("meta", {}).get("date")
    )
    for impact_group in article.get("features", {}).get("llm_drought_impacts") or []:
        for impact in impact_group.get("impacts") or []:
            if impact.get("classification") == WILDFIRE_CLASS:
                wildfire_rows.append({"publication_date": publication_date})

monthly_llm_wildfires = (
    pl.from_dicts(wildfire_rows)
    .with_columns(
        pl.col("publication_date")
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Datetime, strict=False)
        .dt.truncate("1mo")
        .alias("month")
    )
    .drop_nulls("month")
    .group_by("month")
    .agg(pl.len().alias("wildfire_occurrence_count"))
    .sort("month")
)

print(monthly_llm_wildfires.head(12))

# Compare LLM wildfire mentions with observed wildfires on the same monthly axis.
# Limited to the LLM dataset coverage window (Jan 2018 - Dec 2020).
start_date = pd.Timestamp("2018-01-01")
end_date = pd.Timestamp("2020-12-01")
comparison_months = pd.date_range(start=start_date, end=end_date, freq="MS")

llm_series = (
    monthly_llm_wildfires.to_pandas()
    .set_index("month")["wildfire_occurrence_count"]
    .reindex(comparison_months, fill_value=0)
)
fires_series = (
    monthly_fires_pd.set_index("month")["wildfires_total"]
    .reindex(comparison_months, fill_value=0)
)


def normalize(series):
    series_min = series.min()
    series_max = series.max()
    if series_max == series_min:
        return series * 0
    return (series - series_min) / (series_max - series_min)


comparison_df = pd.DataFrame({
    "month": comparison_months,
    "llm_wildfires_scaled": normalize(llm_series).to_numpy(),
    "dataset_wildfires_scaled": normalize(fires_series).to_numpy(),
    "llm_wildfire_occurrence_count": llm_series.to_numpy(),
    "dataset_wildfire_count": fires_series.to_numpy(),
})

plt.figure(figsize=(14, 6))
plt.plot(
    comparison_df["month"],
    comparison_df["llm_wildfires_scaled"],
    label="LLM Wildfire Occurrence (scaled)",
    color=COLOR_LLM,
    linewidth=2.2,
    linestyle="-",
)
plt.plot(
    comparison_df["month"],
    comparison_df["dataset_wildfires_scaled"],
    label="Wildfire dataset (scaled)",
    color=COLOR_WILDFIRE,
    linewidth=2.2,
    linestyle="--",
)
plt.title("Monthly LLM Wildfire Occurrences vs Observed Wildfires (2018-2020)", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Scaled value (0-1)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
save_fig("monthly_llm_vs_observed_wildfires_2018_2020.png")
plt.show()


## NL geocoded wildfire occurrences vs wildfire dataset


In [ ]:
# Load geocoded impacts and keep NL Wildfire Occurrence rows only.
geocoded_csv_path = PROCESSED / "impacts_geocoded_points_filtered.csv"
if not geocoded_csv_path.exists():
    geocoded_csv_path = PROCESSED / "impacts_geocoded_points.csv"

nl_geocoded_wildfires = (
    pl.read_csv(geocoded_csv_path)
    .filter(
        (pl.col("classification") == "Wildfire Occurrence")
        & (pl.col("geocoded_country_code").str.to_lowercase() == "nl")
    )
)

print(
    f"NL Wildfire Occurrence rows: {nl_geocoded_wildfires.height:,} "
    f"(from {geocoded_csv_path.name})"
)

monthly_nl_geocoded_wildfires = (
    nl_geocoded_wildfires
    .with_columns(
        pl.col("publication_date")
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Datetime, strict=False)
        .dt.truncate("1mo")
        .alias("month")
    )
    .drop_nulls("month")
    .group_by("month")
    .agg(pl.len().alias("wildfire_occurrence_count"))
    .sort("month")
)

print(monthly_nl_geocoded_wildfires.head(12))

# Compare NL geocoded wildfire mentions with observed wildfires on the same monthly axis.
# Limited to the geocoded dataset coverage window (Jan 2018 - Dec 2020).
start_date = pd.Timestamp("2018-01-01")
end_date = pd.Timestamp("2020-12-01")
comparison_months = pd.date_range(start=start_date, end=end_date, freq="MS")

nl_series = (
    monthly_nl_geocoded_wildfires.to_pandas()
    .set_index("month")["wildfire_occurrence_count"]
    .reindex(comparison_months, fill_value=0)
)
fires_series = (
    monthly_fires_pd.set_index("month")["wildfires_total"]
    .reindex(comparison_months, fill_value=0)
)


def normalize(series):
    series_min = series.min()
    series_max = series.max()
    if series_max == series_min:
        return series * 0
    return (series - series_min) / (series_max - series_min)


comparison_df = pd.DataFrame({
    "month": comparison_months,
    "nl_geocoded_wildfires_scaled": normalize(nl_series).to_numpy(),
    "dataset_wildfires_scaled": normalize(fires_series).to_numpy(),
    "nl_geocoded_wildfire_count": nl_series.to_numpy(),
    "dataset_wildfire_count": fires_series.to_numpy(),
})

plt.figure(figsize=(14, 6))
plt.plot(
    comparison_df["month"],
    comparison_df["nl_geocoded_wildfires_scaled"],
    label="NL Geocoded Wildfire Occurrence (scaled)",
    color=COLOR_LLM,
    linewidth=2.2,
    linestyle="-",
)
plt.plot(
    comparison_df["month"],
    comparison_df["dataset_wildfires_scaled"],
    label="Wildfire dataset (scaled)",
    color=COLOR_WILDFIRE,
    linewidth=2.2,
    linestyle="--",
)
plt.title("Monthly NL Geocoded Wildfire Occurrences vs Observed Wildfires (2018-2020)", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Scaled value (0-1)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
save_fig("monthly_geocoded_vs_observed_wildfires_2018_2020.png")
plt.show()


## Combined NL geocoded wildfire mentions vs wildfire dataset


In [ ]:
# Load geocoded impacts and combine NL Wildfire Occurrence + Wildfire Risk Increase.
geocoded_csv_path = PROCESSED / "impacts_geocoded_points_filtered.csv"
if not geocoded_csv_path.exists():
    geocoded_csv_path = PROCESSED / "impacts_geocoded_points.csv"

WILDFIRE_CLASSES = ["Wildfire Occurrence", "Wildfire Risk Increase"]

combined_nl_wildfires = (
    pl.read_csv(geocoded_csv_path)
    .filter(
        pl.col("classification").is_in(WILDFIRE_CLASSES)
        & (pl.col("geocoded_country_code").str.to_lowercase() == "nl")
    )
)

class_breakdown = (
    combined_nl_wildfires
    .group_by("classification")
    .agg(pl.len().alias("row_count"))
    .sort("classification")
)
print(f"Combined NL wildfire rows: {combined_nl_wildfires.height:,}")
print(class_breakdown)

monthly_combined_nl_wildfires = (
    combined_nl_wildfires
    .with_columns(
        pl.col("publication_date")
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Datetime, strict=False)
        .dt.truncate("1mo")
        .alias("month")
    )
    .drop_nulls("month")
    .group_by("month")
    .agg(pl.len().alias("combined_wildfire_count"))
    .sort("month")
)

print(monthly_combined_nl_wildfires.head(12))

# Compare combined NL geocoded wildfire mentions with observed wildfires.
# Limited to the geocoded dataset coverage window (Jan 2018 - Dec 2020).
start_date = pd.Timestamp("2018-01-01")
end_date = pd.Timestamp("2020-12-01")
comparison_months = pd.date_range(start=start_date, end=end_date, freq="MS")

combined_series = (
    monthly_combined_nl_wildfires.to_pandas()
    .set_index("month")["combined_wildfire_count"]
    .reindex(comparison_months, fill_value=0)
)
fires_series = (
    monthly_fires_pd.set_index("month")["wildfires_total"]
    .reindex(comparison_months, fill_value=0)
)


def normalize(series):
    series_min = series.min()
    series_max = series.max()
    if series_max == series_min:
        return series * 0
    return (series - series_min) / (series_max - series_min)


comparison_df = pd.DataFrame({
    "month": comparison_months,
    "combined_nl_wildfires_scaled": normalize(combined_series).to_numpy(),
    "dataset_wildfires_scaled": normalize(fires_series).to_numpy(),
    "combined_nl_wildfire_count": combined_series.to_numpy(),
    "dataset_wildfire_count": fires_series.to_numpy(),
})

pearson_r = combined_series.corr(fires_series)
print(f"Pearson correlation (raw monthly counts): {pearson_r:.3f}")

plt.figure(figsize=(14, 6))
plt.plot(
    comparison_df["month"],
    comparison_df["combined_nl_wildfires_scaled"],
    label="Combined NL Geocoded Wildfire Mentions (scaled)",
    color=COLOR_LLM,
    linewidth=2.2,
    linestyle="-",
)
plt.plot(
    comparison_df["month"],
    comparison_df["dataset_wildfires_scaled"],
    label="Wildfire dataset (scaled)",
    color=COLOR_WILDFIRE,
    linewidth=2.2,
    linestyle="--",
)
plt.title("Combined NL Geocoded Wildfire Mentions vs Observed Wildfires (2018-2020)", fontsize=TITLE_FS)
plt.xlabel("Month")
plt.ylabel("Scaled value (0-1)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
save_fig("monthly_combined_geocoded_vs_observed_wildfires_2018_2020.png")
plt.show()


In [ ]:
# Calculate the total count within the specific date range using Polars
total_articles_polars = (
    monthly_articles
    .filter(
        (pl.col("month") >= pl.datetime(2017, 1, 1)) &
        (pl.col("month") <= pl.datetime(2022, 1, 1))
    )
    .select(pl.col("article_count").sum())
    .item()
)

print(f"Total articles between 2017 and 2022 (Polars): {total_articles_polars}")


In [ ]:
# Calculate the total count within the specific date range using Polars
total_articles_polars_2018_2020 = (
    monthly_articles
    .filter(
        (pl.col("month") >= pl.datetime(2018, 1, 1)) &
        (pl.col("month") <= pl.datetime(2020, 1, 1))
    )
    .select(pl.col("article_count").sum())
    .item()
)

print(f"Total articles between 2018 and 2020 (Polars): {total_articles_polars_2018_2020}")


## Interactive monthly map

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Parse month from the full dataset (same logic as the monthly aggregation cell).
if "time" in df_collected.columns:
    df_by_month = df_collected.with_columns(
        pl.col("time")
        .cast(pl.Utf8, strict=False)
        .str.strptime(pl.Datetime, strict=False)
        .dt.truncate("1mo")
        .alias("month")
    )
elif {"year", "month"}.issubset(df_collected.columns):
    df_by_month = df_collected.with_columns(
        pl.date(
            pl.col("year").cast(pl.Int32, strict=False),
            pl.col("month").cast(pl.Int32, strict=False),
            pl.lit(1),
        )
        .cast(pl.Datetime, strict=False)
        .alias("month")
    )
else:
    raise ValueError("Could not find a usable time column. Expected 'time' or 'year'/'month'.")

months = df_by_month.select("month").unique().sort("month")["month"].to_list()
month_labels = [m.strftime("%B %Y") for m in months]
fire_vmax = df_collected["wildfires_sum"].max()


def plot_wildfires_for_month(month_idx):
    selected_month = months[month_idx]
    df_month = df_by_month.filter(pl.col("month") == selected_month)

    plt.close("all")
    plt.figure(figsize=(12, 10))

    df_no_fire = df_month.filter(pl.col("wildfires_sum") == 0)
    plt.scatter(
        df_no_fire["x"],
        df_no_fire["y"],
        color="#E0E0E0",
        s=2,
        alpha=0.3,
        label="No Fire",
    )

    df_fires = df_month.filter(pl.col("wildfires_sum") > 0)
    scatter = plt.scatter(
        df_fires["x"],
        df_fires["y"],
        c=df_fires["wildfires_sum"],
        cmap="YlOrRd",
        vmin=0,
        vmax=fire_vmax,
        s=25,
        alpha=1.0,
        edgecolors="black",
        linewidths=0.4,
    )

    plt.colorbar(scatter, label="Wildfires (Sum)")
    plt.title(
        f"Spatial Distribution of Wildfires in the NL - {month_labels[month_idx]}",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("X Coordinate (Rijksdriehoek / Dutch Grid)")
    plt.ylabel("Y Coordinate")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.show()


month_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(months) - 1,
    step=1,
    description="Month",
    continuous_update=False,
)
month_label = widgets.Label(value=month_labels[0])


def on_month_change(change):
    idx = change["new"]
    month_label.value = month_labels[idx]
    plot_wildfires_for_month(idx)


month_slider.observe(on_month_change, names="value")
display(widgets.VBox([month_slider, month_label]))
plot_wildfires_for_month(0)
